# DeepVF benchmark

In [1]:
# library
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from Bio import SeqIO
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

import src

tqdm.pandas()

# directory
root = Path.cwd()
base_dir = os.path.join(root, "data", "DeepVF")
os.chdir(base_dir)

/home/djmoon/miniforge3/envs/VFTextSeq/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def preprocess(fasta_path, label_path, desc_path, tax_path, id_field="id", replace=True):
    df = src.fasta_to_df(fasta_path)
    label = pd.read_csv(label_path).drop(columns=["Unnamed: 0"]) # drop index
    df = df.merge(label)

    # interproscan
    scan_df = src.parse_interproscan_file(desc_path)
    desc_cols = ["Signature Description", "InterPro Description"]
    description_dict = src.build_interpro_description(scan_df, "id", desc_cols)
    df["desc"] = df["id"].map(lambda x: description_dict.get(str(x), ""))
    model = SentenceTransformer("all-MiniLM-L6-v2")
    df["desc_nodup"] = df["desc"].progress_apply(
        lambda x: src.remove_semantic_duplicates_from_pipe_separated(x, model, 0.85)
    )

    # taxonomy
    tax_df = src.parse_mmseqs_lca(tax_path)
    tax_cols = df["id"].apply(lambda x: map_by_substring(x, tax_df))
    df["taxid"] = tax_cols.apply(lambda x: x[0])
    df["rank"] = tax_cols.apply(lambda x: x[1])
    df["name"] = tax_cols.apply(lambda x: x[2])

    if replace:
        mask = df["sequence"].str.contains("J|\\*", regex=True, na=False)
        print("rows with J or *:", mask.sum())
        df.loc[mask, "sequence"] = df.loc[mask, "sequence"].str.replace("J|\\*", "X", regex=True)
    
    df = df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

    return df

train = preprocess("train.fasta", "train_labels.csv", "virulent_output.tsv", "alnRes_lca_gtdb.tsv")
val   = preprocess("val.fasta",   "val_labels.csv", "virulent_output.tsv", "alnRes_lca_gtdb.tsv")
test  = preprocess("test.fasta",  "test_labels.csv", "virulent_output.tsv", "alnRes_lca_gtdb.tsv")

In [2]:
def preprocess(fasta_path, desc_path, tax_path, id_field="id", replace=True):
    df = src.fasta_to_df(fasta_path)

    # interproscan
    scan_df = pd.read_csv(desc_path)
    df = df.merge(scan_df)

    # taxonomy
    tax_df = pd.read_csv(tax_path)
    df = df.merge(tax_df)

    if replace:
        mask = df["sequence"].str.contains("J|\\*", regex=True, na=False)
        print("rows with J or *:", mask.sum())
        df.loc[mask, "sequence"] = df.loc[mask, "sequence"].str.replace("J|\\*", "X", regex=True)
    
    df = df.drop(columns=["taxid"]) # drop index
    df = df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

    return df

train_pos = preprocess("DeepVF_Training_Dataset/train_positive.fasta", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")
train_neg = preprocess("DeepVF_Training_Dataset/train_negative.fasta", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")
test_pos  = preprocess("DeepVF_Independent_Dataset/ind_positive.fasta", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")
test_neg  = preprocess("DeepVF_Independent_Dataset/ind_negative.fasta", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")

train_pos["label"] = 1.0
test_pos["label"]  = 1.0
train_neg["label"] = 0.0
test_neg["label"]  = 0.0

rows with J or *: 0
rows with J or *: 0
rows with J or *: 0
rows with J or *: 0


In [3]:
train = pd.concat([train_pos, train_neg])
test  = pd.concat([test_pos, test_neg])

In [8]:
X_train, y_train = src.esm_extract(train)
X_test,  y_test  = src.esm_extract(test)

Using cache found in /home/djmoon/.cache/torch/hub/facebookresearch_esm_main
100%|██████████| 7334/7334 [22:43<00:00,  5.38it/s]  
Using cache found in /home/djmoon/.cache/torch/hub/facebookresearch_esm_main


7334


100%|██████████| 1152/1152 [03:33<00:00,  5.39it/s]


1152


In [4]:
from transformers import AutoTokenizer, AutoModel

def bert_extract(df):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
    model = AutoModel.from_pretrained("google-bert/bert-base-uncased").to(device)
    model.eval()
    txt_emb = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting"):
        desc = str(row.get('desc_nodup', '')).strip()
        name = str(row.get('name', '')).strip()
        rank = str(row.get('rank', '')).strip()
        combined_text = f"{desc}|{name}|{rank}"
        embedding = src.get_embedding(combined_text, tokenizer, model, device)
        txt_emb.append(embedding.numpy().tolist())
    
    return np.array(txt_emb)

In [24]:
X_train_text = bert_extract(train)
X_test_text  = bert_extract(test)

Extracting: 100%|██████████| 1152/1152 [00:06<00:00, 189.08it/s]


In [12]:
# np.save("embedding/bert_train.npy", X_train_text)
# np.save("embedding/bert_test.npy", X_test_text)

# reduced duplicates

In [13]:
X_train = np.load("embedding/esm_train.npy")
X_train_text = np.load("embedding/bert_train.npy")
X_train_clip = np.load("proteinclip_emb_esm2/esm_train.npy")
X_train_prot = np.load("embedding/prot_train.npy")

X_test = np.load("embedding/esm_test.npy")
X_test_text = np.load("embedding/bert_test.npy")
X_test_clip = np.load("proteinclip_emb_esm2/esm_test.npy")
X_test_prot = np.load("embedding/prot_test.npy")

# X_train_all = np.concatenate([X_train, X_train_text], axis=1)
# X_test_all  = np.concatenate([X_test, X_test_text], axis=1)

X_train_all = np.concatenate([X_train, X_train_text, X_train_clip, X_train_prot], axis=1)
X_test_all  = np.concatenate([X_test, X_test_text, X_test_clip, X_test_prot], axis=1)

In [11]:
y_train = train.label.values
y_test = test.label.values

print(y_train.shape)

if y_train.ndim != 1:
    y_train = np.asarray(y_train).reshape(-1).astype(int)
    y_test  = np.asarray(y_test).reshape(-1).astype(int)

(7334,)


In [7]:
import warnings
warnings.filterwarnings(action='ignore', category=FutureWarning, module='xgboost')

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0]
}

# Initialize XGBClassifier
xgb = XGBClassifier(
    tree_method="hist",
    eval_metric="auc",
    random_state=42,
)

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5, scoring='roc_auc', verbose=1, n_jobs=-1, error_score="raise")

# Fit grid search
grid_search.fit(X_train_all, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation AUC: ", grid_search.best_score_)

# Evaluate on test data using best estimator
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_all)
y_proba = best_model.predict_proba(X_test_all)[:, 1]

Fitting 5 folds for each of 108 candidates, totalling 540 fits


Best parameters found:  {'colsample_bytree': 0.7, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}

In [14]:
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef, confusion_matrix

best_params = {
    'colsample_bytree': 0.7,
    'learning_rate': 0.2,
    'max_depth': 7,
    'n_estimators': 200,
    'subsample': 1.0,
}

from xgboost import XGBClassifier

best_model = XGBClassifier(
    **best_params,
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
)

best_model.fit(X_train_all, y_train)
y_pred = best_model.predict(X_test_all)
y_proba = best_model.predict_proba(X_test_all)[:, 1]

print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")
tn, fp, fn, tp = (confusion_matrix(y_test, y_pred)).ravel().tolist()
print("specificity", tn/(tn+fp))

Test ROC AUC: 0.9365957754629629
Classification Report:
               precision    recall  f1-score   support

         0.0       0.80      0.94      0.86       576
         1.0       0.92      0.76      0.84       576

    accuracy                           0.85      1152
   macro avg       0.86      0.85      0.85      1152
weighted avg       0.86      0.85      0.85      1152

Accuracy: 0.850
F1-score: 0.836
Precision: 0.922
Recall: 0.764
MCC: 0.710
specificity 0.9357638888888888


In [39]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9513618106305582
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.90      0.89      1442
           1       0.90      0.87      0.89      1442

    accuracy                           0.89      2884
   macro avg       0.89      0.89      0.89      2884
weighted avg       0.89      0.89      0.89      2884

Accuracy: 0.887
F1-score: 0.885
Precision: 0.897
Recall: 0.873
MCC: 0.774


In [29]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9431626689699351
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.89      0.88      1442
           1       0.88      0.86      0.87      1442

    accuracy                           0.88      2884
   macro avg       0.88      0.88      0.88      2884
weighted avg       0.88      0.88      0.88      2884

Accuracy: 0.876
F1-score: 0.874
Precision: 0.884
Recall: 0.865
MCC: 0.751


In [10]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9419555691067077
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.89      0.88      1442
           1       0.88      0.87      0.87      1442

    accuracy                           0.88      2884
   macro avg       0.88      0.88      0.88      2884
weighted avg       0.88      0.88      0.88      2884

Accuracy: 0.876
F1-score: 0.875
Precision: 0.884
Recall: 0.866
MCC: 0.753


In [ ]:
import joblib

# After training and grid search as before
best_model = grid_search.best_estimator_

# Save the model to a file
joblib.dump(best_model, 'best_binary_xgb_model.pkl')

# Later, you can load it back with
# loaded_model = joblib.load('best_xgb_model.pkl')


In [ ]:
import pandas as pd
import joblib

# Load the trained model
loaded_model = joblib.load('best_binary_xgb_model.pkl')

# Load your test set (replace 'test.csv' with your actual test file path)
df_test = pd.read_csv('binary/test_labels.csv')

labels = df_test['label']

# Predict
probabilities = loaded_model.predict_proba(X_test)[:,1]
predictions = loaded_model.predict(X_test)


# Prepare result dataframe with id, predicted label, and original label
result_df = pd.DataFrame({
    'id': df_test['id'],
    'prob': probabilities,
    'pred': predictions,
    'label': labels
})

# Save results
result_df.to_csv('binary/VFTextSeq_prediction_best_xgb.csv', index=False)


In [ ]:
result_df